# M4 V2 — Điều tra cả 3 seed (42, 43, 44)
**Nguồn:** `RECOVERED_ORIGINAL_V2_TEST_12_MODELS_20261016.zip` (bộ kết quả test sau M4). **CPU only**; không train/inference.

Mục tiêu: (1) so sánh các nhánh theo seed; (2) kiểm tra F1/Recall/Precision của 16 lớp; (3) xem lớp bị nhầm sang đâu; (4) đối chiếu chính xác các mẫu Learned-local sửa đúng/làm sai so với Random-local và Original GAN. **Quan sát trên test chỉ dùng giải thích hậu kiểm, không chọn lại mô hình.**

In [ ]:
from pathlib import Path
import zipfile, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Kaggle: add ZIP as dataset trước khi Run All; VS Code: đặt ZIP cùng notebook hoặc sửa ZIP_PATH.
ZIP_PATH = None  # ví dụ: '/kaggle/input/ten-dataset/RECOVERED_ORIGINAL_V2_TEST_12_MODELS_20261016.zip'
OUT = Path('/kaggle/working/M4_V2_diagnostics') if Path('/kaggle/working').exists() else Path('M4_V2_diagnostics')
OUT.mkdir(parents=True, exist_ok=True)
SEEDS = [42, 43, 44]
ARMS = ['original_gan', 'matched_ros', 'random_local', 'learned_local']
TARGET_CLASSES = [9, 13, 2, 10]  # sẽ dùng tên thực từ class_labels; có thể thay sau

if ZIP_PATH is None:
    found = []
    for d in [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd(), Path('/mnt/data')]:
        if d.exists():
            found += list(d.rglob('RECOVERED_ORIGINAL_V2_TEST_12_MODELS_20261016.zip'))
    found = sorted(set(found), key=lambda p: len(str(p)))
    if not found:
        raise FileNotFoundError('Chưa thấy ZIP gốc. Tải ZIP lên Kaggle hoặc đặt ZIP_PATH.')
    ZIP_PATH = found[0]
ZIP_PATH = Path(ZIP_PATH)
print('ZIP:', ZIP_PATH, '| OUTPUT:', OUT)
z = zipfile.ZipFile(ZIP_PATH)
names = z.namelist()
def locate(suffix):
    matches = [x for x in names if x.endswith(suffix)]
    if len(matches) != 1:
        raise ValueError(f'Yêu cầu đúng 1 file kết thúc bằng {suffix}; tìm được {len(matches)}: {matches[:5]}')
    return matches[0]
def csv(suffix):
    return pd.read_csv(io.BytesIO(z.read(locate(suffix))))
pred = csv('/analysis/predictions_all_models.csv')
per = csv('/analysis/per_class_by_seed.csv')
metrics = csv('/analysis/metrics_by_seed.csv')
labels = csv('/analysis/class_labels.csv')
print('Loaded:', len(pred), 'predictions;',len(per),'class rows;',len(metrics),'models')
display(labels)


## 1. Kiểm tra tính nhất quán trước khi phân tích

In [ ]:
expected = {(a,s) for a in ARMS for s in SEEDS}
actual = set(map(tuple, metrics[['arm','seed']].to_numpy()))
assert actual == expected, f'Sai số lượng branch×seed: {expected-actual}, {actual-expected}'
assert len(per) == 12*16
assert len(pred) == 12*4721
assert pred.groupby(['arm','seed']).size().eq(4721).all()
assert pred['correct'].eq((pred.y_true_0based == pred.y_pred_0based).astype(int)).all()
ref = pred[(pred.arm=='original_gan') & (pred.seed==42)].sort_values('test_index_0based')
for (a,s), g in pred.groupby(['arm','seed']):
    g = g.sort_values('test_index_0based')
    assert np.array_equal(g.test_index_0based.to_numpy(), ref.test_index_0based.to_numpy()), (a,s,'index')
    for k in ['row_0based','col_0based','y_true_0based','true_class_id_1based']:
        assert np.array_equal(g[k].to_numpy(),ref[k].to_numpy()), (a,s,k)
assert pred.true_class_id_1based.eq(pred.y_true_0based+1).all()
assert pred.pred_class_id_1based.eq(pred.y_pred_0based+1).all()
assert np.isclose(metrics['OA'].mean(), pred.groupby(['arm','seed']).correct.mean().mean(),atol=1e-8)
print('PASS: 12 nhánh×seed, 4721 mẫu/nhánh, cùng thứ tự tọa độ và nhãn, OA khớp')


## 2. Tổng thể và chênh lệch theo seed

In [ ]:
m = metrics.copy()
for k in ['OA','AA','Macro_F1','GM']:
    m[k+'_pct']=100*m[k]
display(m[['arm','seed','OA_pct','AA_pct','Macro_F1_pct','GM_pct']].sort_values(['seed','arm']).round(3))
pivot=m.pivot(index='seed',columns='arm',values='Macro_F1_pct').reindex(SEEDS)
pivot['learned_minus_random_pp']=pivot['learned_local']-pivot['random_local']
pivot['learned_minus_original_pp']=pivot['learned_local']-pivot['original_gan']
display(pivot.round(3))
pivot.to_csv(OUT/'macro_f1_by_seed.csv')
fig, ax=plt.subplots(figsize=(8,4))
for a in ARMS:ax.plot(SEEDS,pivot[a],marker='o',label=a)
ax.set_xticks(SEEDS);ax.set_ylabel('Macro-F1 (%)');ax.set_xlabel('Seed');ax.grid(alpha=.25);ax.legend();fig.tight_layout()
fig.savefig(OUT/'macro_f1_by_seed.png',dpi=160);plt.show()


## 3. Phân tích từng lớp **cả 3 seed**
Chênh lệch F1 (điểm phần trăm) Learned-local trừ Random-local. Giá trị dương/tăng không có nghĩa cải thiện có ý nghĩa thống kê.

In [ ]:
f=per.pivot_table(index=['class_id','class_name'],columns=['seed','arm'],values=['F1','recall','precision'])
rows=[]
for (cid, cname), r in f.iterrows():
    d={'class_id':int(cid),'class_name':cname}
    for s in SEEDS:
        for metric in ['F1','recall','precision']:
            d[f'{metric}_delta_seed{s}_pp']=100*(r[(metric,s,'learned_local')]-r[(metric,s,'random_local')])
    rows.append(d)
delta=pd.DataFrame(rows).sort_values('class_id')
fcols=[f'F1_delta_seed{s}_pp' for s in SEEDS]
delta['F1_delta_mean_pp']=delta[fcols].mean(axis=1)
delta['F1_positive_seeds']=(delta[fcols]>0).sum(axis=1)
display(delta[['class_id','class_name',*fcols,'F1_delta_mean_pp','F1_positive_seeds']].round(2))
delta.to_csv(OUT/'per_class_delta_all_seeds.csv',index=False)
fig,ax=plt.subplots(figsize=(10,7))
heat=delta[fcols].to_numpy()
im=ax.imshow(heat,aspect='auto',cmap='coolwarm',vmin=-max(1,np.abs(heat).max()),vmax=max(1,np.abs(heat).max()))
ax.set_xticks(range(3),[f'Seed {s}' for s in SEEDS]);ax.set_yticks(range(16),delta.class_name)
fig.colorbar(im,ax=ax,label='ΔF1 (điểm %)');fig.tight_layout();fig.savefig(OUT/'f1_class_heatmap.png',dpi=160);plt.show()


## 4. Phân tích mẫu ghép cặp: khi nào V2 sửa đúng hoặc làm sai?
So sánh **cùng test_index/tọa độ** cho mỗi seed. Xuất mẫu đổi kết quả để kiểm tra sau.

In [ ]:
pair_rows=[]; switches=[]
for s in SEEDS:
    sub=pred[pred.seed==s]
    wide=sub.pivot(index='test_index_0based',columns='arm',values='correct')
    for comp in ['random_local','original_gan','matched_ros']:
        win=(wide.learned_local.eq(1)&wide[comp].eq(0))
        lose=(wide.learned_local.eq(0)&wide[comp].eq(1))
        pair_rows.append({'seed':s,'comparison':comp,'V2_right_other_wrong':int(win.sum()),'V2_wrong_other_right':int(lose.sum()),'net_correct':int(win.sum()-lose.sum()),'discordant':int(win.sum()+lose.sum())})
        if comp=='random_local':
            t=sub[sub.arm=='learned_local'][['test_index_0based','row_0based','col_0based','true_class_id_1based','pred_class_id_1based']].rename(columns={'pred_class_id_1based':'pred_learned'})
            r=sub[sub.arm=='random_local'][['test_index_0based','pred_class_id_1based']].rename(columns={'pred_class_id_1based':'pred_random'})
            merged=t.merge(r,on='test_index_0based',validate='one_to_one')
            merged['seed']=s
            merged['outcome']=np.where(merged.pred_learned.eq(merged.true_class_id_1based)&merged.pred_random.ne(merged.true_class_id_1based),'V2_only_correct',np.where(merged.pred_learned.ne(merged.true_class_id_1based)&merged.pred_random.eq(merged.true_class_id_1based),'Random_only_correct','same_correctness'))
            switches.append(merged[merged.outcome!='same_correctness'])
pairs=pd.DataFrame(pair_rows)
display(pairs)
pairs.to_csv(OUT/'paired_correctness_by_seed.csv',index=False)
switches=pd.concat(switches,ignore_index=True)
switches.to_csv(OUT/'discordant_samples_learned_vs_random.csv',index=False)
display(switches.groupby(['seed','true_class_id_1based','outcome']).size().unstack(fill_value=0).head(30))


## 5. Confusion matrix: so sánh theo seed và lớp
Ma trận: **hàng = nhãn thật, cột = nhãn dự đoán**. Thống kê nhầm lẫn theo lớp để xem V2 đang chuyển lỗi sang lớp nào.

In [ ]:
name_by_id=dict(zip(labels.class_id_1based,labels.class_name))
cm_dict={};errors=[]
for (a,s),g in pred.groupby(['arm','seed']):
    mat=np.zeros((16,16),dtype=np.int64)
    np.add.at(mat,(g.y_true_0based.to_numpy(),g.y_pred_0based.to_numpy()),1)
    cm_dict[(a,int(s))]=mat
    for ci in range(16):
        wrong=[(name_by_id[j+1],int(mat[ci,j])) for j in range(16) if j!=ci and mat[ci,j]>0]
        wrong.sort(key=lambda x:-x[1])
        errors.append({'arm':a,'seed':int(s),'class_id':ci+1,'class_name':name_by_id[ci+1], 'support':int(mat[ci].sum()),'correct':int(mat[ci,ci]),'mistakes':int(mat[ci].sum()-mat[ci,ci]),'top_wrong_destination':wrong[0][0] if wrong else '', 'top_wrong_count':wrong[0][1] if wrong else 0})
err=pd.DataFrame(errors)
err.to_csv(OUT/'confusion_error_summary.csv',index=False)
for s in SEEDS:
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    for ax,a in zip(axes,['learned_local','random_local']):
        cm=cm_dict[(a,s)]; norm=cm/np.maximum(cm.sum(axis=1,keepdims=True),1)
        im=ax.imshow(norm,vmin=0,vmax=1,cmap='Blues');ax.set_title(f'{a} | seed {s}')
        ax.set_xticks(range(16));ax.set_yticks(range(16));ax.set_xticklabels(range(1,17),fontsize=7);ax.set_yticklabels(range(1,17),fontsize=7);ax.set_xlabel('Predicted class');ax.set_ylabel('True class')
    fig.colorbar(im,ax=axes.ravel().tolist(),shrink=.7,label='Row-normalized fraction')
    fig.savefig(OUT/f'confusion_seed_{s}.png',dpi=150,bbox_inches='tight');plt.show()


## 6. Lớp đáng chú ý trên 3 seed
Tự động chọn lớp biến động mạnh về chênh lệch F1 (không chọn mô hình). Với từng lớp, cho biết nhầm sang đâu và bao nhiêu mẫu.

In [ ]:
important=delta.assign(span=delta[fcols].max(axis=1)-delta[fcols].min(axis=1)).sort_values('span',ascending=False).head(6)
display(important[['class_id','class_name','span',*fcols]].round(2))
for cid in important.class_id:
    print('\n'+'='*70,'Class',cid,name_by_id[cid])
    display(err[(err.class_id==cid)&(err.arm.isin(['learned_local','random_local']))][['seed','arm','support','correct','mistakes','top_wrong_destination','top_wrong_count']].sort_values(['seed','arm']))


## 7. Tổng hợp, lưu sản phẩm và giới hạn suy luận
- Kết quả xác định **những lớp/mẫu có vấn đề**, chưa xác định nguyên nhân bên trong GAN.
- Không khẳng định seed 43 tệ do học trọng số khi chưa có log/ablation riêng.
- Nếu muốn nghiên cứu loss/hành vi weight, cần log **M4 training**; không thể suy ngược từ confusion matrix.
- Tuyệt đối không dùng test này để chọn hyperparameter rồi báo cáo lại như test độc lập.

In [ ]:
summary={
 'model_count':len(metrics), 'test_per_model':4721,
 'seed_comparisons':pairs.to_dict(orient='records'),
 'positive_F1_all_three_seeds':delta.loc[delta.F1_positive_seeds==3,['class_id','class_name']].to_dict(orient='records'),
 'methodological_note':'Post-hoc test diagnostics only; no causal attribution or model selection.'
}
import json
(OUT/'analysis_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf8')
print('Đã xuất vào:',OUT.resolve())
for p in sorted(OUT.iterdir()): print(' -',p.name)
